# Trial Decision Tree - Drop + Impute + Log1p + Tuning
**Goal:** Compare a Decision Tree against the Random Forest pipeline with the same preprocessing style.

In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


RANDOM_STATE = 42
TARGET = "Survived"
np.random.seed(RANDOM_STATE)

In [15]:
# Load data
df = pd.read_csv("titanic_augmented.csv")

# Quick sanity checks
print("Shape:", df.shape)
print(df[TARGET].value_counts(normalize=True))

Shape: (891, 26)
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [16]:
# Feature engineering
if 'title' in df.columns:
    title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
    df['title_grouped'] = df['title'].map(title_mapping).fillna('Rare')

if 'SibSp' in df.columns and 'Parch' in df.columns:
    df['is_alone'] = ((df['SibSp'] == 0) & (df['Parch'] == 0)).astype(int)

if 'Age' in df.columns:
    df['age_group'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                             labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])
    df['age_group'] = df['age_group'].astype(str)

if 'Fare' in df.columns:
    df['fare_group'] = pd.qcut(df['Fare'].fillna(df['Fare'].median()),
                               q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'],
                               duplicates='drop')
    df['fare_group'] = df['fare_group'].astype(str)

if 'Cabin' in df.columns:
    df['cabin_deck'] = df['Cabin'].str[0].fillna('Unknown')

df['age_missing'] = df['Age'].isna().astype(int)
df['cabin_missing'] = df['Cabin'].isna().astype(int)
if 'Embarked' in df.columns:
    df['embarked_missing'] = df['Embarked'].isna().astype(int)
print("Feature engineering complete!")
print("New shape:", df.shape)

Feature engineering complete!
New shape: (891, 32)


## Preprocessing Design
- Drop high-cardinality or ID-like columns
- Impute missing values
- Log-transform a few skewed numeric columns
- One-hot encode categorical columns

In [17]:
# Columns to drop for efficiency
# (high-cardinality, mostly-missing, or ID-like)
drop_cols = [
    "Name", "Ticket", "Cabin", "cabin_room_number",
    "PassengerId", "service_id", "booking_reference",
    "title",   
    "Age",     
    "Fare", 
]

X = df.drop(columns=[TARGET] + [c for c in drop_cols if c in df.columns])
y = df[TARGET]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# Log-transform candidates (only keep those that exist)
# Removed "Fare" since we dropped it (using fare_group instead)
log_cols = [c for c in ["fare_per_person", "ticket_group_size", "family_size", "SibSp", "Parch"] if c in num_cols]
num_cols_no_log = [c for c in num_cols if c not in log_cols]

print("Numeric:", num_cols)
print("Categorical:", cat_cols)
print("Log cols:", log_cols)

Numeric: ['Pclass', 'SibSp', 'Parch', 'name_length', 'family_size', 'is_alone', 'ticket_group_size', 'fare_per_person', 'age_fare_ratio', 'cabin_score', 'name_word_count', 'age_missing', 'cabin_missing', 'embarked_missing']
Categorical: ['Sex', 'Embarked', 'title_group', 'cabin_deck', 'title_grouped', 'age_group', 'fare_group']
Log cols: ['fare_per_person', 'ticket_group_size', 'family_size', 'SibSp', 'Parch']


In [18]:
# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(668, 21) (223, 21)
Survived
0    0.616766
1    0.383234
Name: proportion, dtype: float64
Survived
0    0.61435
1    0.38565
Name: proportion, dtype: float64


In [19]:
# Preprocessing pipeline
log_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

# OneHotEncoder compatibility across sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("log_num", log_pipeline, log_cols),
        ("num", num_pipeline, num_cols_no_log),
        ("cat", cat_pipeline, cat_cols),
    ],
    remainder="drop"
)

## Baseline Model (Optional)

In [20]:
# Baseline Decision Tree
dt_base = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", dt_base)
])

baseline_model.fit(X_train, y_train)
y_pred_base = baseline_model.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, y_pred_base))
print(confusion_matrix(y_test, y_pred_base))

Baseline accuracy: 0.7309417040358744
[[111  26]
 [ 34  52]]


## Hyperparameter Tuning (Grid Search)
Use stratified 5-fold CV on the training set only.

In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    "model__max_depth": [4, 6, 8, 10, None],          
    "model__min_samples_split": [2, 10, 20],          
    "model__min_samples_leaf": [1, 4, 8],             
    "model__max_features": [None, "sqrt"],            
    "model__class_weight": [None, "balanced"],        
    "model__criterion": ["gini", "entropy"],          
    "model__ccp_alpha": [0.0, 0.01],                  
}

dt = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

dt_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", dt)
])

grid = GridSearchCV(
    dt_model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

Fitting 5 folds for each of 720 candidates, totalling 3600 fits
Best params: {'model__ccp_alpha': 0.0, 'model__class_weight': None, 'model__criterion': 'entropy', 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
Best CV accuracy: 0.8203905285602066


## Final Evaluation on Test Set

In [22]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

# Feature count after preprocessing
feature_count = best_model.named_steps["preprocess"].transform(X_train).shape[1]
print("Final feature count:", feature_count)

Test accuracy: 0.7757847533632287

Confusion matrix:
 [[114  23]
 [ 27  59]]

Classification report:
               precision    recall  f1-score   support

           0       0.81      0.83      0.82       137
           1       0.72      0.69      0.70        86

    accuracy                           0.78       223
   macro avg       0.76      0.76      0.76       223
weighted avg       0.77      0.78      0.77       223

Final feature count: 48


## Feature Importance (Top 10)

In [23]:
importances = best_model.named_steps["model"].feature_importances_

# If feature names are available
try:
    feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    print(fi.head(10))
except Exception:
    print("Feature names not available. Showing top 10 importances only:")
    print(np.sort(importances)[-10:][::-1])

num__name_word_count          0.182751
cat__title_grouped_Miss       0.168020
cat__title_group_Mr           0.135854
log_num__fare_per_person      0.089759
num__Pclass                   0.079926
num__age_fare_ratio           0.063115
num__cabin_missing            0.051187
cat__Embarked_Q               0.046805
log_num__ticket_group_size    0.044469
cat__fare_group_Low           0.025792
dtype: float64


## Save Decision Tree Results

In [24]:
trial_dt_results = {
    "drop_cols": drop_cols,
    "log_cols": log_cols,
    "best_params": grid.best_params_,
    "best_cv_accuracy": grid.best_score_,
    "test_accuracy": accuracy_score(y_test, y_pred),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "classification_report": classification_report(y_test, y_pred, output_dict=True),
    "final_feature_count": int(feature_count),
}

trial_dt_results

{'drop_cols': ['Name',
  'Ticket',
  'Cabin',
  'cabin_room_number',
  'PassengerId',
  'service_id',
  'booking_reference',
  'title',
  'Age',
  'Fare'],
 'log_cols': ['fare_per_person',
  'ticket_group_size',
  'family_size',
  'SibSp',
  'Parch'],
 'best_params': {'model__ccp_alpha': 0.0,
  'model__class_weight': None,
  'model__criterion': 'entropy',
  'model__max_depth': 8,
  'model__max_features': 'sqrt',
  'model__min_samples_leaf': 4,
  'model__min_samples_split': 2},
 'best_cv_accuracy': np.float64(0.8203905285602066),
 'test_accuracy': 0.7757847533632287,
 'confusion_matrix': [[114, 23], [27, 59]],
 'classification_report': {'0': {'precision': 0.8085106382978723,
   'recall': 0.8321167883211679,
   'f1-score': 0.8201438848920863,
   'support': 137.0},
  '1': {'precision': 0.7195121951219512,
   'recall': 0.686046511627907,
   'f1-score': 0.7023809523809523,
   'support': 86.0},
  'accuracy': 0.7757847533632287,
  'macro avg': {'precision': 0.7640114167099117,
   'recall': 0.